# 01 — EDA dos pluviômetros CEMADEN de Petrópolis

**Objetivo (Fase 2):** entender os dados antes de qualquer modelo. Este notebook responde às
perguntas abertas das pesquisas:

| Pergunta | Origem |
|---|---|
| Qual a resolução temporal real (10 min? horária?) | [003](../docs/research/003-inventario-de-dados.md) |
| Quantas estações têm série contínua? Quanta lacuna? | 003 |
| Qual o volume do tombo (menor valor > 0)? | [004](../docs/research/004-controle-de-qualidade-pluviometros.md) |
| Quantos eventos ≥10/20/40 mm em 1 h por estação? | [002](../docs/research/002-definicao-chuva-destrutiva.md), ADR 0004 |
| Janela de 1 h móvel ou hora cheia? | ADR 0004 |
| Os eventos de 15/02/2022 e 20/03/2022 aparecem nos dados? | 002, 004 |

Dados: *Dados da Rede Observacional do CEMADEN/MCTI* (citação obrigatória).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_DIR = Path("../data/raw/cemaden")
files = sorted(RAW_DIR.glob("**/*.csv"))
print(f"{len(files)} arquivos CSV encontrados")
files[:10]

## 1. Inspecionar o formato bruto
Antes de carregar com pandas, olhamos o texto cru: separador, encoding, cabeçalho, formato de data.
Nunca confie no formato antes de ver o arquivo.

In [ ]:
amostra = files[0]
with open(amostra, "rb") as f:
    raw = f.read(2000)
print(raw.decode("utf-8", errors="replace"))

**Preencha após a inspeção acima** — nomes de colunas e formato reais do CSV do PED.

In [ ]:
SEP = ";"            # ajustar: ";" ou ","
ENCODING = "utf-8"   # ajustar se aparecerem caracteres estranhos (ex.: "latin-1")
COL_TIME = "datahora"        # ajustar
COL_VALUE = "valorMedida"    # ajustar — chuva em mm
COL_STATION = "codEstacao"   # ajustar
TZ_DADOS = "UTC"     # confirmar: os horários vêm em UTC ou horário local?

## 2. Carregar todas as estações em um único DataFrame (formato longo)

In [ ]:
def carregar(path):
    df = pd.read_csv(path, sep=SEP, encoding=ENCODING)
    df[COL_TIME] = pd.to_datetime(df[COL_TIME])
    return df

df = pd.concat([carregar(p) for p in files], ignore_index=True)
df = df.rename(columns={COL_TIME: "time", COL_VALUE: "rain_mm", COL_STATION: "station"})
df = df.sort_values(["station", "time"]).reset_index(drop=True)
df.info()
df.head()

## 3. Inventário por estação
- **Período:** primeira e última leitura.
- **Frequência de registro:** moda do intervalo entre leituras (esperado: 10 min).
- **Duplicatas:** mesmo instante repetido (erro estrutural, ver 004).
- **Cobertura:** % de intervalos esperados que têm dado.

Atenção: estações CEMADEN podem só enviar registro quando chove — nesse caso, "sem registro"
pode significar zero, não lacuna. Verificar antes de concluir.

In [ ]:
def inventario(g):
    dt = g["time"].diff().dropna()
    freq = dt.mode().iloc[0] if len(dt) else pd.NaT
    esperado = (g["time"].max() - g["time"].min()) / freq + 1 if pd.notna(freq) else np.nan
    return pd.Series({
        "inicio": g["time"].min(),
        "fim": g["time"].max(),
        "n_registros": len(g),
        "freq_moda": freq,
        "duplicatas": g["time"].duplicated().sum(),
        "cobertura_%": 100 * len(g) / esperado if esperado else np.nan,
        "max_mm": g["rain_mm"].max(),
    })

inv = df.groupby("station").apply(inventario).sort_values("inicio")
inv

### 3.1 Linha do tempo de disponibilidade
Cada linha é uma estação; cada ponto, um dia com algum registro. Mostra visualmente quando cada
estação esteve ativa e onde há buracos.

In [ ]:
dias = df.assign(dia=df["time"].dt.floor("D")).groupby(["station", "dia"]).size().reset_index()
fig, ax = plt.subplots(figsize=(12, 0.3 * df["station"].nunique() + 1))
for i, (st, g) in enumerate(dias.groupby("station")):
    ax.plot(g["dia"], np.full(len(g), i), "|", ms=4)
ax.set_yticks(range(dias["station"].nunique()), dias["station"].unique())
ax.set_title("Dias com registro por estação")
plt.tight_layout()

## 4. Resolução do pluviômetro (volume do tombo)
O menor valor positivo e os valores mais comuns revelam a resolução do instrumento (ex.: 0,2 mm).
Valores que não são múltiplos dessa resolução podem indicar processamento ou erro.

In [ ]:
pos = df.loc[df["rain_mm"] > 0, "rain_mm"]
print("menor valor > 0:", pos.min())
pos.round(3).value_counts().head(15)

## 5. Controle de qualidade
⏸️ **Aguardando decisão** sobre a pesquisa [004](../docs/research/004-controle-de-qualidade-pluviometros.md).
Regra já fixada: dados brutos nunca são alterados; o QC adiciona uma coluna `qc_flag`.

## 6. Catálogo de eventos (ADR 0004: 10 / 20 / 40 mm em 1 h)
Comparamos duas definições de "chuva em 1 h":
- **hora cheia:** soma de 13h00–14h00;
- **janela móvel:** soma dos últimos 60 min a cada leitura — captura picos que cruzam a virada da hora.

⚠️ Antes do QC, estes números são **preliminares**.

In [ ]:
LIMIARES = [10, 20, 40]

def acum_1h(g):
    s = g.set_index("time")["rain_mm"].sort_index()
    s = s[~s.index.duplicated()]
    return pd.DataFrame({
        "movel_1h": s.rolling("60min").sum(),
        "hora_cheia": s.resample("1h").sum().reindex(s.index, method="ffill"),
    })

acc = df.groupby("station", group_keys=True).apply(acum_1h).reset_index()

def n_eventos(serie, limiar):
    # conta "eventos" (blocos contíguos acima do limiar), não leituras individuais
    acima = serie >= limiar
    return int((acima & ~acima.shift(fill_value=False)).sum())

linhas = []
for st, g in acc.groupby("station"):
    for L in LIMIARES:
        linhas.append({"station": st, "limiar": L,
                       "movel": n_eventos(g["movel_1h"], L),
                       "hora_cheia": n_eventos(g["hora_cheia"], L)})
cat = pd.DataFrame(linhas).pivot(index="station", columns="limiar")
cat

In [ ]:
# Totais da rede e por ano — responde: há exemplos suficientes de >=40 mm/h para treinar?
acc["ano"] = acc["time"].dt.year
por_ano = {L: acc.groupby("ano").apply(lambda g: sum(n_eventos(x["movel_1h"], L) for _, x in g.groupby("station")))
           for L in LIMIARES}
pd.DataFrame(por_ano)

In [ ]:
# Top 20 maiores acumulados de 1 h (janela móvel) — candidatos a revisão manual no QC
acc.nlargest(20, "movel_1h")[["station", "time", "movel_1h"]]

## 7. Quando chove forte? Sazonalidade e ciclo diurno
Esperado (a confirmar): concentração entre novembro e março, à tarde/noite (convecção).
Atenção ao fuso: converter para horário local antes de interpretar o ciclo diurno.

In [ ]:
fortes = acc[acc["movel_1h"] >= 20].copy()
t_local = fortes["time"].dt.tz_localize(TZ_DADOS).dt.tz_convert("America/Sao_Paulo")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_local.dt.month.value_counts().sort_index().plot.bar(ax=axes[0], title="Leituras ≥20 mm/1h por mês")
t_local.dt.hour.value_counts().sort_index().plot.bar(ax=axes[1], title="Leituras ≥20 mm/1h por hora local")
plt.tight_layout()

## 8. Casos conhecidos — teste de sanidade
Os dados precisam mostrar os eventos documentados (Alcântara et al., 2023):
- **15/02/2022:** ~258 mm em ~3 h (São Sebastião);
- **20/03/2022:** 415 mm em 10 h (São Sebastião).

In [ ]:
def plot_evento(inicio, fim, top=6):
    janela = acc[(acc["time"] >= inicio) & (acc["time"] <= fim)]
    principais = janela.groupby("station")["movel_1h"].max().nlargest(top).index
    fig, ax = plt.subplots(figsize=(12, 4))
    for st in principais:
        g = janela[janela["station"] == st]
        ax.plot(g["time"], g["movel_1h"], label=st)
    for L in LIMIARES:
        ax.axhline(L, ls="--", lw=0.8, c="gray")
    ax.set_ylabel("mm na última 1 h"); ax.legend(); ax.set_title(f"{inicio} → {fim}")

plot_evento("2022-02-15 12:00", "2022-02-16 06:00")

In [ ]:
plot_evento("2022-03-20 00:00", "2022-03-21 12:00")

## 9. Conclusões (preencher após rodar)
- Resolução temporal: …
- Estações utilizáveis: …
- Nº de eventos por limiar: … → manter ou ajustar ADR 0004?
- Janela 1 h escolhida: …
- Problemas de qualidade observados: … → alimenta o QC (pesquisa 004)